# NIH metrics

This notebook uses information extracted from https://reporter.nih.gov/exporter to identify NIH-funded users of PhysioNet.

## Setup

In [3]:
import os
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from rapidfuzz.distance import JaroWinkler

from twentyfiveyears.nih import (nih_exporter_table_combined, standardize_nih_exporter_project_names,
                                 standardize_nih_exporter_publication_names, get_nih_exporter_project_leader_names,
                                 get_nih_exporter_publication_authors, best_jaro_winkler_match, check_nih_exporter_hits)

In [38]:
# Set the base path
base_path = os.path.join("..", "data")
# Set the required variables
physionet_users = os.path.join(base_path, 'physionet', 'users.csv')
person_id = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
nih_exporter_projects_folder = os.path.join(base_path, 'nih', 'exporter', 'projects')
nih_exporter_publications_folder = os.path.join(base_path, 'nih', 'exporter', 'publications')
nih_exporter_start_year = 1995
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')


Run the main script

In [45]:
# Get a DataFrame of all of the PhysioNet users
df_pn_users = pd.read_csv(physionet_users)
# Get the PhysioNet user_id to person_id mapping table
df_person_id_mapping = pd.read_csv(person_id)
# Add person_id column to the users table
df_pn_users = pd.merge(df_pn_users, df_person_id_mapping, left_on='user_id', right_on='physionet_id')
#Only keep the columns we need from the users table
df_pn_users = df_pn_users[['person_id', 'full_name']].copy()

# Check to see if PhysioNet users can be found in the NIH exporter data
df_pn_users = check_nih_exporter_hits(df_pn_users, nih_exporter_projects_folder, nih_exporter_publications_folder,nih_exporter_start_year)

# Rename the 'full_name' column to 'physionet_name'
df_pn_users = df_pn_users.rename(columns={'full_name': 'physionet_name'})

/var/folders/7s/ddpxs9cs7t11m92j93r_tbfm0000gn/T/ipykernel_71958/1632615556.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pn_users = pd.read_csv(physionet_users)


# user names = 10


/var/folders/7s/ddpxs9cs7t11m92j93r_tbfm0000gn/T/ipykernel_71958/314477523.py:10: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(folder, pattern + str(year) + '.csv'), encoding_errors='replace')
/var/folders/7s/ddpxs9cs7t11m92j93r_tbfm0000gn/T/ipykernel_71958/314477523.py:10: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(folder, pattern + str(year) + '.csv'), encoding_errors='replace')
/var/folders/7s/ddpxs9cs7t11m92j93r_tbfm0000gn/T/ipykernel_71958/314477523.py:10: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(folder, pattern + str(year) + '.csv'), encoding_errors='replace')
/var/folders/7s/ddpxs9cs7t11m92j93r_tbfm0000gn/T/ipykernel_71958/314477523.py:10: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import

# project_formatted_names = 231693
# publication_formatted_names = 3053124
Searching for PhysioNet users who are PIs that were awarded grants in NIH Explorer...


100%|██████████| 10/10 [00:00<00:00, 83.94it/s]


Searching for PhysioNet users who are published authors from NIH Explorer...


100%|██████████| 10/10 [00:00<00:00, 9650.95it/s]


Finished in 9.327073097229004 seconds


Save the results

In [39]:
# Save the results
path = Path(save_path)
# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()
# Output the merged DataFrame or save it to a file
df_pn_users.to_csv(normalized_path, index=False)